# 05 Model Evaluation

Notebook ini digunakan untuk tahap kelima sistem Case-Based Reasoning (CBR), yaitu mengevaluasi performa sistem retrieval dan prediksi solusi.

Input yang digunakan:
- File hasil tahap 4: `stage_04_solution_reuse_output.zip`
- Dataset kasus: `data/processed/cases.csv`
- Query evaluasi: `data/eval/queries.json`
- Model retrieval: `data/processed/tfidf_vectorizer.pkl`
- Matriks TF-IDF: `data/processed/tfidf_matrix.npz`

Evaluasi yang dilakukan:
1. Evaluasi retrieval:
   - Accuracy@1
   - Accuracy@3
   - Accuracy@5
   - Precision@5
   - Recall@5
   - F1@5
   - Mean Reciprocal Rank atau MRR

2. Evaluasi prediksi solusi:
   - Accuracy
   - Precision
   - Recall
   - F1-score

Output utama:
- `data/results/retrieval_metrics.csv`
- `data/results/retrieval_eval_details.csv`
- `data/results/prediction_metrics.csv`
- `data/results/prediction_eval_details.csv`
- `stage_05_evaluation_output.zip`


## 1. Instalasi Library

Cell ini memasang library yang dibutuhkan untuk evaluasi retrieval dan prediksi.


In [ ]:
!pip -q install pandas numpy scikit-learn scipy joblib tqdm openpyxl

## 2. Import Library

Cell ini memanggil library yang digunakan untuk membaca data, memuat model, menghitung similarity, dan menghitung metrik evaluasi.


In [ ]:
import os
import re
import json
import shutil
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy import sparse
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

print("Library berhasil dimuat.")

## 3. Menyiapkan Struktur Folder

Cell ini menyiapkan struktur folder project dan memastikan folder output tersedia.


In [ ]:
BASE_DIR = Path("/content/cbr-putusan-penganiayaan")

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder project siap digunakan.")
print(BASE_DIR)

## 4. Upload dan Ekstrak Output Tahap 4

Cell ini meminta file `stage_04_solution_reuse_output.zip`. File tersebut berisi dataset, model retrieval, query evaluasi, dan hasil prediksi dari tahap sebelumnya.


In [ ]:
EXPECTED_ZIP_NAME = "stage_04_solution_reuse_output.zip"
zip_path = Path("/content") / EXPECTED_ZIP_NAME

if not zip_path.exists():
    try:
        from google.colab import files
        print("Silakan upload file stage_04_solution_reuse_output.zip")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("Tidak ada file yang diupload.")
        zip_path = Path("/content") / list(uploaded.keys())[0]
    except Exception as e:
        raise RuntimeError(
            "File ZIP tahap 4 belum tersedia. Upload file stage_04_solution_reuse_output.zip terlebih dahulu."
        ) from e

print(f"File ZIP digunakan: {zip_path}")

extract_temp = Path("/content/stage_04_extract_temp")
if extract_temp.exists():
    shutil.rmtree(extract_temp)
extract_temp.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_temp)

candidate_data_dirs = list(extract_temp.rglob("data"))

if len(candidate_data_dirs) == 0:
    raise FileNotFoundError("Folder data tidak ditemukan di dalam ZIP tahap 4.")

source_root = candidate_data_dirs[0].parent

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
shutil.copytree(source_root, BASE_DIR)

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output tahap 4 berhasil diekstrak.")

## 5. Memuat Dataset, Query Evaluasi, dan Model

Cell ini memuat dataset `cases.csv`, query evaluasi `queries.json`, model TF-IDF, dan matriks TF-IDF yang diperlukan untuk menghitung ulang hasil retrieval.


In [ ]:
cases_csv_path = PROCESSED_DIR / "cases.csv"
queries_path = EVAL_DIR / "queries.json"
vectorizer_path = PROCESSED_DIR / "tfidf_vectorizer.pkl"
tfidf_matrix_path = PROCESSED_DIR / "tfidf_matrix.npz"

required_files = [cases_csv_path, queries_path, vectorizer_path, tfidf_matrix_path]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

cases_df = pd.read_csv(cases_csv_path)

with open(queries_path, "r", encoding="utf-8") as f:
    queries = json.load(f)

vectorizer = joblib.load(vectorizer_path)
tfidf_matrix = sparse.load_npz(tfidf_matrix_path)

print("Dataset, query, dan model berhasil dimuat.")
print("Jumlah case:", len(cases_df))
print("Jumlah query evaluasi:", len(queries))
print("Ukuran TF-IDF matrix:", tfidf_matrix.shape)

display(cases_df[["case_id", "nomor_perkara", "pasal", "amar_label"]].head())
queries[:2]

## 6. Fungsi Preprocessing dan Retrieval

Cell ini mendefinisikan kembali fungsi preprocessing dan retrieval agar evaluasi dapat dilakukan secara mandiri pada notebook tahap 5.


In [ ]:
def preprocess_text(text):
    """
    Membersihkan teks agar konsisten dengan tahap retrieval sebelumnya.
    """
    if not isinstance(text, str):
        text = ""

    text = text.lower()

    text = text.replace("kuhpidana", "kuhp")
    text = text.replace("kuh pidana", "kuhp")
    text = text.replace("pidanaidana", "pidana")
    text = text.replace("nopember", "november")
    text = text.replace("pebruari", "februari")

    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()

    return text


def retrieve(query: str, k: int = 5, exclude_case_id: str = None):
    """
    Mengambil top-k kasus paling mirip berdasarkan TF-IDF dan cosine similarity.
    """
    clean_query = preprocess_text(query)
    query_vector = vectorizer.transform([clean_query])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    result_df = cases_df.copy()
    result_df["similarity_score"] = similarities

    if exclude_case_id is not None:
        result_df = result_df[result_df["case_id"] != exclude_case_id]

    result_df = result_df.sort_values(by="similarity_score", ascending=False)

    selected_columns = [
        "case_id",
        "nomor_perkara",
        "tanggal_putusan",
        "terdakwa",
        "pasal",
        "amar_label",
        "similarity_score",
        "ringkasan_fakta",
        "amar_putusan"
    ]

    available_columns = [col for col in selected_columns if col in result_df.columns]

    return result_df[available_columns].head(k).reset_index(drop=True)


## 7. Evaluasi Retrieval

Cell ini mengevaluasi apakah ground truth case muncul pada hasil top-k retrieval. Karena setiap query dibuat dari kasus tertentu, ground truth-nya adalah `ground_truth_case_id` pada file `queries.json`.


In [ ]:
retrieval_eval_rows = []

for query in tqdm(queries, desc="Evaluasi retrieval"):
    query_id = query["query_id"]
    query_text = query["query_text"]
    ground_truth_case_id = query["ground_truth_case_id"]
    ground_truth_nomor_perkara = query.get("ground_truth_nomor_perkara", "")

    results = retrieve(query_text, k=5)

    top_case_ids = results["case_id"].tolist()
    top_nomor_perkara = results["nomor_perkara"].tolist()
    top_scores = results["similarity_score"].round(6).tolist()

    found_at_1 = ground_truth_case_id in top_case_ids[:1]
    found_at_3 = ground_truth_case_id in top_case_ids[:3]
    found_at_5 = ground_truth_case_id in top_case_ids[:5]

    if found_at_5:
        rank = top_case_ids.index(ground_truth_case_id) + 1
        reciprocal_rank = 1 / rank
    else:
        rank = None
        reciprocal_rank = 0

    # Untuk satu ground truth relevan per query:
    # Precision@5 = 1/5 jika ground truth muncul di top-5, jika tidak 0
    # Recall@5 = 1 jika ground truth muncul di top-5, jika tidak 0
    precision_at_5 = 1 / 5 if found_at_5 else 0
    recall_at_5 = 1 if found_at_5 else 0

    if precision_at_5 + recall_at_5 == 0:
        f1_at_5 = 0
    else:
        f1_at_5 = 2 * (precision_at_5 * recall_at_5) / (precision_at_5 + recall_at_5)

    retrieval_eval_rows.append({
        "query_id": query_id,
        "ground_truth_case_id": ground_truth_case_id,
        "ground_truth_nomor_perkara": ground_truth_nomor_perkara,
        "top_5_case_ids": ", ".join(top_case_ids),
        "top_5_nomor_perkara": ", ".join(top_nomor_perkara),
        "top_5_similarity_scores": ", ".join(map(str, top_scores)),
        "ground_truth_found_at_1": found_at_1,
        "ground_truth_found_at_3": found_at_3,
        "ground_truth_found_at_5": found_at_5,
        "ground_truth_rank": rank,
        "reciprocal_rank": reciprocal_rank,
        "precision_at_5": precision_at_5,
        "recall_at_5": recall_at_5,
        "f1_at_5": f1_at_5
    })

retrieval_eval_df = pd.DataFrame(retrieval_eval_rows)
retrieval_eval_df

## 8. Ringkasan Metrik Retrieval

Cell ini menghitung rata-rata metrik retrieval, kemudian menyimpannya ke `retrieval_metrics.csv`.


In [ ]:
retrieval_metrics = {
    "accuracy_at_1": retrieval_eval_df["ground_truth_found_at_1"].mean(),
    "accuracy_at_3": retrieval_eval_df["ground_truth_found_at_3"].mean(),
    "accuracy_at_5": retrieval_eval_df["ground_truth_found_at_5"].mean(),
    "precision_at_5": retrieval_eval_df["precision_at_5"].mean(),
    "recall_at_5": retrieval_eval_df["recall_at_5"].mean(),
    "f1_at_5": retrieval_eval_df["f1_at_5"].mean(),
    "mrr": retrieval_eval_df["reciprocal_rank"].mean(),
    "jumlah_query": len(retrieval_eval_df)
}

retrieval_metrics_df = pd.DataFrame([retrieval_metrics])

retrieval_metrics_path = RESULTS_DIR / "retrieval_metrics.csv"
retrieval_eval_details_path = RESULTS_DIR / "retrieval_eval_details.csv"

retrieval_metrics_df.to_csv(retrieval_metrics_path, index=False)
retrieval_eval_df.to_csv(retrieval_eval_details_path, index=False)

print("Metrik retrieval:")
display(retrieval_metrics_df)

print("File disimpan:")
print("-", retrieval_metrics_path)
print("-", retrieval_eval_details_path)

## 9. Fungsi Ekstraksi Lama Pidana dan Prediksi Outcome

Cell ini mendefinisikan fungsi untuk mengekstrak lama pidana dari amar putusan, melakukan majority vote label amar, dan memprediksi outcome dari query evaluasi.


In [ ]:
number_words = {
    "satu": 1,
    "dua": 2,
    "tiga": 3,
    "empat": 4,
    "lima": 5,
    "enam": 6,
    "tujuh": 7,
    "delapan": 8,
    "sembilan": 9,
    "sepuluh": 10,
    "sebelas": 11,
    "dua belas": 12,
    "tiga belas": 13,
    "empat belas": 14,
    "lima belas": 15,
    "enam belas": 16,
    "tujuh belas": 17,
    "delapan belas": 18,
    "sembilan belas": 19,
    "dua puluh": 20,
    "tiga puluh": 30,
}


def word_to_number(value):
    if value is None:
        return None

    value = str(value).lower().strip()

    if value.isdigit():
        return int(value)

    value = re.sub(r"[^a-z\\s]", " ", value)
    value = re.sub(r"\\s+", " ", value).strip()

    return number_words.get(value)


def extract_sentence_months(amar_text):
    """
    Mengekstrak lama pidana penjara dari amar putusan dalam satuan bulan.
    """
    if not isinstance(amar_text, str):
        return None

    text = amar_text.lower()
    text = re.sub(r"\\s+", " ", text)

    match_section = re.search(r"pidana\\s+penjara\\s+selama\\s+(.*?)(?:;|\\.|,| menetapkan| membebankan)", text)
    if match_section:
        section = match_section.group(1)
    else:
        section = text

    total_months = 0
    found = False

    year_patterns = [
        r"(\\d+)\\s*\\([^)]*\\)\\s*tahun",
        r"([a-z\\s]+?)\\s*tahun"
    ]

    for pattern in year_patterns:
        match = re.search(pattern, section)
        if match:
            value = match.group(1).strip()
            num = word_to_number(value)
            if num is not None:
                total_months += num * 12
                found = True
                break

    month_patterns = [
        r"(\\d+)\\s*\\([^)]*\\)\\s*bulan",
        r"([a-z\\s]+?)\\s*bulan"
    ]

    for pattern in month_patterns:
        match = re.search(pattern, section)
        if match:
            value = match.group(1).strip()
            words = value.split()
            if len(words) > 3:
                value = " ".join(words[-2:])
            num = word_to_number(value)
            if num is not None:
                total_months += num
                found = True
                break

    if found:
        return int(total_months)

    return None


def majority_vote_label(top_k_df):
    if "amar_label" not in top_k_df.columns:
        return "tidak_diketahui"

    counts = top_k_df["amar_label"].value_counts()

    if len(counts) == 0:
        return "tidak_diketahui"

    top_count = counts.max()
    candidate_labels = counts[counts == top_count].index.tolist()

    if len(candidate_labels) == 1:
        return candidate_labels[0]

    for _, row in top_k_df.sort_values("similarity_score", ascending=False).iterrows():
        if row["amar_label"] in candidate_labels:
            return row["amar_label"]

    return candidate_labels[0]


def predict_outcome(query: str, k: int = 5):
    """
    Memprediksi label amar menggunakan majority vote dari top-k kasus termirip.
    """
    top_k = retrieve(query, k=k)
    predicted_label = majority_vote_label(top_k)

    top_1 = top_k.iloc[0]

    return {
        "predicted_label": predicted_label,
        "recommended_solution_from_case_id": top_1["case_id"],
        "recommended_solution_from_nomor_perkara": top_1["nomor_perkara"],
        "top_k_cases": top_k
    }


cases_df["sentence_months"] = cases_df["amar_putusan"].apply(extract_sentence_months)
cases_df[["case_id", "nomor_perkara", "amar_label", "sentence_months"]].head()


## 10. Evaluasi Prediksi Solusi

Cell ini mengevaluasi prediksi `amar_label` dari query evaluasi. Label aktual diambil dari case ground truth, sedangkan label prediksi diperoleh dari fungsi `predict_outcome()`.


In [ ]:
prediction_eval_rows = []

case_label_map = dict(zip(cases_df["case_id"], cases_df["amar_label"]))
case_nomor_map = dict(zip(cases_df["case_id"], cases_df["nomor_perkara"]))

for query in tqdm(queries, desc="Evaluasi prediksi solusi"):
    query_id = query["query_id"]
    query_text = query["query_text"]
    ground_truth_case_id = query["ground_truth_case_id"]

    actual_label = case_label_map.get(ground_truth_case_id, "tidak_diketahui")

    pred = predict_outcome(query_text, k=5)
    predicted_label = pred["predicted_label"]
    top_k_df = pred["top_k_cases"]

    prediction_eval_rows.append({
        "query_id": query_id,
        "ground_truth_case_id": ground_truth_case_id,
        "ground_truth_nomor_perkara": case_nomor_map.get(ground_truth_case_id, ""),
        "actual_label": actual_label,
        "predicted_label": predicted_label,
        "is_correct": actual_label == predicted_label,
        "recommended_solution_from_case_id": pred["recommended_solution_from_case_id"],
        "recommended_solution_from_nomor_perkara": pred["recommended_solution_from_nomor_perkara"],
        "top_5_case_ids": ", ".join(top_k_df["case_id"].tolist()),
        "top_5_similarity_scores": ", ".join(map(str, top_k_df["similarity_score"].round(6).tolist()))
    })

prediction_eval_df = pd.DataFrame(prediction_eval_rows)
prediction_eval_df

## 11. Ringkasan Metrik Prediksi

Cell ini menghitung accuracy, precision, recall, dan F1-score untuk prediksi label amar.


In [ ]:
y_true = prediction_eval_df["actual_label"]
y_pred = prediction_eval_df["predicted_label"]

prediction_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    "jumlah_query": len(prediction_eval_df)
}

prediction_metrics_df = pd.DataFrame([prediction_metrics])

prediction_metrics_path = RESULTS_DIR / "prediction_metrics.csv"
prediction_eval_details_path = RESULTS_DIR / "prediction_eval_details.csv"
classification_report_path = RESULTS_DIR / "classification_report.txt"

prediction_metrics_df.to_csv(prediction_metrics_path, index=False)
prediction_eval_df.to_csv(prediction_eval_details_path, index=False)

report_text = classification_report(y_true, y_pred, zero_division=0)

with open(classification_report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Metrik prediksi:")
display(prediction_metrics_df)

print("\nClassification report:")
print(report_text)

print("File disimpan:")
print("-", prediction_metrics_path)
print("-", prediction_eval_details_path)
print("-", classification_report_path)

## 12. Analisis Hasil Evaluasi

Cell ini membuat ringkasan analisis secara otomatis berdasarkan hasil retrieval dan prediksi.


In [ ]:
analysis_text = f'''
Analisis Hasil Evaluasi

1. Evaluasi Retrieval
Sistem retrieval menggunakan metode TF-IDF dan cosine similarity. Berdasarkan {len(retrieval_eval_df)} query evaluasi, diperoleh:
- Accuracy@1: {retrieval_metrics["accuracy_at_1"]:.4f}
- Accuracy@3: {retrieval_metrics["accuracy_at_3"]:.4f}
- Accuracy@5: {retrieval_metrics["accuracy_at_5"]:.4f}
- Precision@5: {retrieval_metrics["precision_at_5"]:.4f}
- Recall@5: {retrieval_metrics["recall_at_5"]:.4f}
- F1@5: {retrieval_metrics["f1_at_5"]:.4f}
- MRR: {retrieval_metrics["mrr"]:.4f}

Nilai tersebut menunjukkan kemampuan sistem dalam menemukan kembali kasus ground truth pada daftar top-k hasil retrieval.

2. Evaluasi Prediksi Solusi
Prediksi solusi dilakukan dengan mengambil label amar dari top-5 kasus termirip menggunakan majority vote. Berdasarkan {len(prediction_eval_df)} query evaluasi, diperoleh:
- Accuracy: {prediction_metrics["accuracy"]:.4f}
- Precision Macro: {prediction_metrics["precision_macro"]:.4f}
- Recall Macro: {prediction_metrics["recall_macro"]:.4f}
- F1 Macro: {prediction_metrics["f1_macro"]:.4f}

3. Kesimpulan
Sistem CBR berhasil membangun case base dari 30 putusan, merepresentasikan kasus ke dalam bentuk fitur teks, melakukan retrieval kasus menggunakan TF-IDF dan cosine similarity, menggunakan kembali solusi dari kasus lama, serta mengevaluasi hasil retrieval dan prediksi dengan metrik yang terukur.
'''

analysis_path = RESULTS_DIR / "evaluation_analysis.txt"

with open(analysis_path, "w", encoding="utf-8") as f:
    f.write(analysis_text.strip())

print(analysis_text)
print("Analisis disimpan di:", analysis_path)

## 13. Membuat Ringkasan Tahap 5

Cell ini membuat file ringkasan tahap 5 yang berisi lokasi seluruh output evaluasi.


In [ ]:
stage_05_summary = {
    "jumlah_case_base": len(cases_df),
    "jumlah_query_evaluasi": len(queries),
    "retrieval_method": "TF-IDF + Cosine Similarity",
    "prediction_method": "Majority vote from top-5 similar cases",
    "retrieval_accuracy_at_5": retrieval_metrics["accuracy_at_5"],
    "prediction_accuracy": prediction_metrics["accuracy"],
    "output_retrieval_metrics": str(retrieval_metrics_path),
    "output_retrieval_eval_details": str(retrieval_eval_details_path),
    "output_prediction_metrics": str(prediction_metrics_path),
    "output_prediction_eval_details": str(prediction_eval_details_path),
    "output_analysis": str(analysis_path)
}

stage_05_summary_df = pd.DataFrame([stage_05_summary])
stage_05_summary_path = PROCESSED_DIR / "stage_05_summary.csv"
stage_05_summary_df.to_csv(stage_05_summary_path, index=False)

stage_05_summary_df

## 14. Membuat ZIP Output Tahap 5

Cell ini membuat ZIP project setelah tahap 5 selesai. ZIP ini berisi seluruh output dari tahap 1 sampai tahap 5 dan siap digunakan untuk penyusunan repository GitHub.


In [ ]:
output_zip = Path("/content/stage_05_evaluation_output.zip")

if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(
    base_name=str(output_zip).replace(".zip", ""),
    format="zip",
    root_dir=BASE_DIR
)

print(f"ZIP output tahap 5 berhasil dibuat: {output_zip}")

## Kesimpulan Tahap 5

Pada tahap ini, sistem CBR telah dievaluasi menggunakan metrik retrieval dan metrik prediksi. Evaluasi retrieval menghasilkan nilai Accuracy@k, Precision@5, Recall@5, F1@5, dan MRR. Evaluasi prediksi menghasilkan Accuracy, Precision, Recall, dan F1-score. Output evaluasi ini dapat digunakan sebagai bukti performa sistem pada laporan dan repository GitHub.
